# 小样本学习 Few-Shot Learning

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

小样本学习旨在从极少的标注样本中学习新类别，模拟人类从少量示例快速学习的能力。

Few-shot learning aims to learn new categories from very few labeled samples, mimicking human ability to learn quickly from few examples.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/fewshot.png" width=500>

# 概述 Overview

* **目标:**  从少量样本（1-5个）中识别新类别。
* **优点:** 
  * 减少标注成本
  * 模拟人类学习能力
  * 适应动态类别需求
* **缺点:**
  * 容易过拟合
  * 训练不稳定
  * 对噪声敏感
* **其他:** 
  * N-way K-shot 任务设定
  * 广泛应用于目标检测、图像分类、NLP

# 设置 Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances

# 原型网络 Prototypical Networks

In [ ]:
# 原型网络通过计算类原型（均值）进行分类
# Prototypical Networks classify by computing class prototypes (mean)

class PrototypicalNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(PrototypicalNetwork, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
    
    def forward(self, x):
        return self.encoder(x)
    
    def compute_prototypes(self, support_x, support_y, n_classes):
        # 计算每个类的原型（均值）
        prototypes = []
        for c in range(n_classes):
            mask = support_y == c
            class_samples = support_x[mask]
            prototype = class_samples.mean(dim=0)
            prototypes.append(prototype)
        return torch.stack(prototypes)

# 小样本任务生成 Few-Shot Task Generation

In [ ]:
# 生成小样本分类任务
# N-way K-shot: N classes, K samples per class for support

def create_fewshot_task(n_way, k_shot, query_per_class, input_dim):
    """
    创建小样本任务
    - n_way: 类别数
    - k_shot: 每个类的支持样本数
    - query_per_class: 每个类的查询样本数
    """
    n_classes = n_way
    
    # 为每个类生成样本
    support_x_list = []
    support_y_list = []
    query_x_list = []
    query_y_list = []
    
    for c in range(n_classes):
        # 每个类有自己的中心，防止重叠
        center = torch.randn(input_dim) * 3 + c * 5
        
        # 支持集样本
        support_samples = center + torch.randn(k_shot, input_dim) * 0.5
        support_x_list.append(support_samples)
        support_y_list.append(torch.full((k_shot,), c, dtype=torch.long))
        
        # 查询集样本
        query_samples = center + torch.randn(query_per_class, input_dim) * 0.5
        query_x_list.append(query_samples)
        query_y_list.append(torch.full((query_per_class,), c, dtype=torch.long))
    
    support_x = torch.cat(support_x_list, dim=0)
    support_y = torch.cat(support_y_list, dim=0)
    query_x = torch.cat(query_x_list, dim=0)
    query_y = torch.cat(query_y_list, dim=0)
    
    return support_x, support_y, query_x, query_y

# 测试任务生成
n_way = 5
k_shot = 2
query_per_class = 3
input_dim = 20

support_x, support_y, query_x, query_y = create_fewshot_task(
    n_way, k_shot, query_per_class, input_dim
)
print(f"Support set: {support_x.shape}, labels: {support_y.shape}")
print(f"Query set: {query_x.shape}, labels: {query_y.shape}")

# 训练原型网络 Train Prototypical Network

In [ ]:
# 初始化
model = PrototypicalNetwork(input_dim=20, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 训练多个epoch
num_epochs = 100
losses = []

for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # 创建任务
    support_x, support_y, query_x, query_y = create_fewshot_task(
        n_way=5, k_shot=2, query_per_class=3, input_dim=20
    )
    
    # 编码
    support_emb = model(support_x)
    query_emb = model(query_x)
    
    # 计算原型
    prototypes = model.compute_prototypes(support_emb, support_y, n_way)
    
    # 计算距离（负欧氏距离作为相似度）
    # Query to prototypes: (n_query, n_way)
    dists = torch.cdist(query_emb, prototypes)
    
    # 交叉熵损失
    log_probs = F.log_softmax(-dists, dim=1)
    loss = F.nll_loss(log_probs, query_y)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# 可视化训练曲线 Visualize Training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Prototypical Network Training')
plt.grid(True)
plt.show()

# 评估小样本学习 Evaluate Few-Shot Learning

In [ ]:
# 在新任务上评估
def evaluate_fewshot(model, n_way, k_shot, query_per_class, input_dim, n_tasks=20):
    correct = 0
    total = 0
    
    for _ in range(n_tasks):
        support_x, support_y, query_x, query_y = create_fewshot_task(
            n_way, k_shot, query_per_class, input_dim
        )
        
        # 编码
        support_emb = model(support_x)
        query_emb = model(query_x)
        
        # 计算原型
        prototypes = model.compute_prototypes(support_emb, support_y, n_way)
        
        # 预测
        dists = torch.cdist(query_emb, prototypes)
        preds = dists.argmin(dim=1)
        
        correct += (preds == query_y).sum().item()
        total += query_y.size(0)
    
    return correct / total

model.eval()
accuracy = evaluate_fewshot(model, n_way=5, k_shot=2, query_per_class=3, input_dim=20, n_tasks=50)
print(f"5-way 2-shot Accuracy: {accuracy:.2%}")

# TODO

- MAML (Model-Agnostic Meta-Learning)
- 匹配网络 Matching Networks
- 关系网络 Relation Networks
- 数据增强 for Few-Shot